In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Dataframe examples").getOrCreate()
print("Spark session created")

Spark session created


In [3]:
data = [
    (1, "Rahul", "IT", 70000, "2024-01-10"),
    (2, "Sneha", "HR", 60000, "2024-02-15"),
    (3, "Arjun", "IT", 75000, "2024-03-01"),
    (4, "Priya", "Finance", 80000, "2024-01-25"),
    (5, "Karan", None, 50000, "2024-02-20")
]
columns = ["emp_id", "name", "department", "salary", "joining_date"]
df = spark.createDataFrame(data, columns)
df.show()


+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+



In [4]:
df.show()
df.printSchema()
df.columns
df.count()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+

root
 |-- emp_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- joining_date: string (nullable = true)



5

In [5]:
df.show()
df_renamed=df.withColumnRenamed("salary","emp_salary")
df_renamed.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+

+------+-----+----------+----------+------------+
|emp_id| name|department|emp_salary|joining_date|
+------+-----+----------+----------+------------+
|     1|Rahul|        IT|     70000|  2024-01-10|
|     2|Sneha|        HR|     60000|  2024-02-15|
|     3|Arjun|        IT|     75000|  2024-03-01|
|     4|Priya|   Finance|     80000|  2024-01-25|
|     5|Karan|      NULL|     50000|  2024-02-20|
+------+-----+----------+----------+------------+



In [6]:
df.filter(df.department=="IT").show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     3|Arjun|        IT| 75000|  2024-03-01|
+------+-----+----------+------+------------+



In [7]:
df.filter((df.salary>60000)&(df.department=="IT")).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     3|Arjun|        IT| 75000|  2024-03-01|
+------+-----+----------+------+------------+



In [8]:
df.drop("joining_date").show()

+------+-----+----------+------+
|emp_id| name|department|salary|
+------+-----+----------+------+
|     1|Rahul|        IT| 70000|
|     2|Sneha|        HR| 60000|
|     3|Arjun|        IT| 75000|
|     4|Priya|   Finance| 80000|
|     5|Karan|      NULL| 50000|
+------+-----+----------+------+



In [9]:
from pyspark.sql.functions import sum,avg,max,min
df.select(
    sum("salary").alias("total_salary"),
    avg("salary").alias("average_salary"),
    max("salary").alias("max_salary"),
    min("salary").alias("min_salary")
).show()

+------------+--------------+----------+----------+
|total_salary|average_salary|max_salary|min_salary|
+------------+--------------+----------+----------+
|      335000|       67000.0|     80000|     50000|
+------------+--------------+----------+----------+



In [10]:
dept_data = [
    ("IT", "Bangalore"),
    ("HR", "Mumbai"),
    ("Finance", "Delhi")
]

dept_columns = ["department", "location"]

dept_df = spark.createDataFrame(dept_data, dept_columns)
dept_df.show()

+----------+---------+
|department| location|
+----------+---------+
|        IT|Bangalore|
|        HR|   Mumbai|
|   Finance|    Delhi|
+----------+---------+



In [11]:
df.join(dept_df,on="department",how="inner").show()

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|   Finance|     4|Priya| 80000|  2024-01-25|    Delhi|
|        HR|     2|Sneha| 60000|  2024-02-15|   Mumbai|
|        IT|     1|Rahul| 70000|  2024-01-10|Bangalore|
|        IT|     3|Arjun| 75000|  2024-03-01|Bangalore|
+----------+------+-----+------+------------+---------+



In [12]:
df.join(dept_df,on="department",how="left").show()

+----------+------+-----+------+------------+---------+
|department|emp_id| name|salary|joining_date| location|
+----------+------+-----+------+------------+---------+
|        HR|     2|Sneha| 60000|  2024-02-15|   Mumbai|
|        IT|     1|Rahul| 70000|  2024-01-10|Bangalore|
|      NULL|     5|Karan| 50000|  2024-02-20|     NULL|
|   Finance|     4|Priya| 80000|  2024-01-25|    Delhi|
|        IT|     3|Arjun| 75000|  2024-03-01|Bangalore|
+----------+------+-----+------+------------+---------+



In [13]:
df.withColumn("bonus",df.salary*0.1).show()

+------+-----+----------+------+------------+------+
|emp_id| name|department|salary|joining_date| bonus|
+------+-----+----------+------+------------+------+
|     1|Rahul|        IT| 70000|  2024-01-10|7000.0|
|     2|Sneha|        HR| 60000|  2024-02-15|6000.0|
|     3|Arjun|        IT| 75000|  2024-03-01|7500.0|
|     4|Priya|   Finance| 80000|  2024-01-25|8000.0|
|     5|Karan|      NULL| 50000|  2024-02-20|5000.0|
+------+-----+----------+------+------------+------+



In [14]:
from pyspark.sql.functions import col
df.filter(col("department").isNull()).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+



In [15]:
df.na.fill({"department":"Unknown"}).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|   Unknown| 50000|  2024-02-20|
+------+-----+----------+------+------------+



In [16]:
df.na.drop().show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
+------+-----+----------+------+------------+



In [17]:
from pyspark.sql.functions import upper,lower,length
df.select(
    "name",
    upper("name").alias("upper_name"),
    lower("name").alias("lower_name"),
    length("name").alias("name_length")
).show()

+-----+----------+----------+-----------+
| name|upper_name|lower_name|name_length|
+-----+----------+----------+-----------+
|Rahul|     RAHUL|     rahul|          5|
|Sneha|     SNEHA|     sneha|          5|
|Arjun|     ARJUN|     arjun|          5|
|Priya|     PRIYA|     priya|          5|
|Karan|     KARAN|     karan|          5|
+-----+----------+----------+-----------+



In [18]:
from google.colab import files
uploaded=files.upload()

Saving sales.csv to sales.csv
Saving employees.csv to employees.csv
Saving employees_nested.json to employees_nested.json
Saving departments.csv to departments.csv


In [19]:
!ls

departments.csv  employees.csv	employees_nested.json  sales.csv  sample_data


In [20]:
df=spark.read.csv("employees.csv",header=True,inferSchema=True)
df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [21]:
from pyspark.sql.functions import to_date,year,month,dayofmonth
df_dates=df.withColumn("joining_date",to_date("joining_date"))
df_dates.show()


+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
+------+-----+----------+------+------------+



In [22]:
df_dates.select(
    "name",
    "joining_date",
    year("joining_date").alias("year"),
    month("joining_date").alias("month"),
    dayofmonth("joining_date").alias("day")
).show()

+-----+------------+----+-----+---+
| name|joining_date|year|month|day|
+-----+------------+----+-----+---+
|Rahul|  2023-01-10|2023|    1| 10|
|Sneha|  2022-11-15|2022|   11| 15|
|Arjun|  2023-03-01|2023|    3|  1|
|Priya|  2022-12-20|2022|   12| 20|
|Karan|  2023-02-05|2023|    2|  5|
|Meera|  2023-04-10|2023|    4| 10|
| Amit|  2023-01-18|2023|    1| 18|
+-----+------------+----+-----+---+



In [25]:
new_data=[
    (6,"Meera","IT",72000,"2024-04-01"),
    (7,"Amit","HR",50000,"2024-04-10")
]
new_df=spark.createDataFrame(new_data,columns)
new_df.show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     6|Meera|        IT| 72000|  2024-04-01|
|     7| Amit|        HR| 50000|  2024-04-10|
+------+-----+----------+------+------------+



In [26]:
df.union(new_df).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
|     6|Meera|        IT| 72000|  2024-04-01|
|     7| Amit|        HR| 50000|  2024-04-10|
+------+-----+----------+------+------------+



In [27]:
df.unionByName(new_df).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2023-01-10|
|     2|Sneha|        HR| 60000|  2022-11-15|
|     3|Arjun|        IT| 75000|  2023-03-01|
|     4|Priya|   Finance| 80000|  2022-12-20|
|     5|Karan|        IT| 50000|  2023-02-05|
|     6|Meera|      NULL| 72000|  2023-04-10|
|     7| Amit|        HR| 58000|  2023-01-18|
|     6|Meera|        IT| 72000|  2024-04-01|
|     7| Amit|        HR| 50000|  2024-04-10|
+------+-----+----------+------+------------+



In [28]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
def salary_band(salary):
  if salary>=75000:
    return "High"
  elif salary>=60000:
    return "Medium"
  else:
    return "Low"
salary_band_udf=udf(salary_band,StringType())
df.withColumn("salary_band",salary_band_udf("salary")).show()

+------+-----+----------+------+------------+-----------+
|emp_id| name|department|salary|joining_date|salary_band|
+------+-----+----------+------+------------+-----------+
|     1|Rahul|        IT| 70000|  2023-01-10|     Medium|
|     2|Sneha|        HR| 60000|  2022-11-15|     Medium|
|     3|Arjun|        IT| 75000|  2023-03-01|       High|
|     4|Priya|   Finance| 80000|  2022-12-20|       High|
|     5|Karan|        IT| 50000|  2023-02-05|        Low|
|     6|Meera|      NULL| 72000|  2023-04-10|     Medium|
|     7| Amit|        HR| 58000|  2023-01-18|        Low|
+------+-----+----------+------+------------+-----------+

